In [ ]:
!pip install torch torchvision

In [ ]:
import os
import torch
import cv2
import kagglehub
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


In [ ]:
downloaded_data_dir = kagglehub.dataset_download("andrewmvd/car-plate-detection")
print(f"Downloaded to {downloaded_data_dir}")

images_dir = os.path.join(downloaded_data_dir, "images")
annotations_dir = os.path.join(downloaded_data_dir, "annotations")


In [ ]:
class LicensePlateDataset(Dataset):
    def __init__(self, images_dir, annotations_dir):
        self.images_dir = images_dir
        self.annotations_dir = annotations_dir
        self.image_files = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)

        # Load image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Load XML annotation
        xml_name = img_name.replace(".png", ".xml").replace(".jpg", ".xml")
        xml_path = os.path.join(self.annotations_dir, xml_name)

        if not os.path.exists(xml_path):
            return self.__getitem__((idx + 1) % len(self))

        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):
            xmin = int(obj.find("bndbox/xmin").text)
            ymin = int(obj.find("bndbox/ymin").text)
            xmax = int(obj.find("bndbox/xmax").text)
            ymax = int(obj.find("bndbox/ymax").text)

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(1)  # license plate class

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        # Convert image
        img = torch.as_tensor(img, dtype=torch.float32) / 255.0
        img = img.permute(2, 0, 1)

        return img, target


In [ ]:
dataset = LicensePlateDataset(images_dir, annotations_dir)

train_size = int(0.75 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)


In [ ]:
model = fasterrcnn_resnet50_fpn(pretrained=True)

num_classes = 2  # background + license plate

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")

print("Training complete!")


In [ ]:
model.eval()
images, _ = next(iter(val_loader))
images = [img.to(device) for img in images]

with torch.no_grad():
    predictions = model(images)

print(predictions[0])